In [ ]:
# ============================================================
# ONEHEALTH NEXUS
# COMPLETE DATA COLLECTION + PREPROCESSING PIPELINE
# ============================================================

import os
import re
import time
import json
import shutil
import zipfile
import requests
import pandas as pd
import xml.etree.ElementTree as ET
from html import unescape
from google.colab import files

# ============================================================
# 1. PROJECT FOLDERS
# ============================================================

BASE_DIR = "/content/OneHealth_Nexus"
RAW_DIR = os.path.join(BASE_DIR, "data", "raw")
PROCESSED_DIR = os.path.join(BASE_DIR, "data", "processed")

os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

print("=" * 70)
print("        ONEHEALTH NEXUS - DATA PIPELINE")
print("=" * 70)

print("\n[1/8] Project folders created.")
print("BASE:", BASE_DIR)


# ============================================================
# 2. HELPER FUNCTIONS
# ============================================================

def clean_text(text):
    """
    Clean text for NLP preprocessing.
    """
    if text is None:
        return ""

    text = str(text)

    # Remove HTML/XML tags
    text = re.sub(r"<[^>]+>", " ", text)

    # Decode HTML entities
    text = unescape(text)

    # Replace newlines/tabs with spaces
    text = re.sub(r"[\r\n\t]+", " ", text)

    # Normalize multiple spaces
    text = re.sub(r"\s+", " ", text)

    # Keep letters, numbers and basic punctuation
    text = re.sub(r"[^A-Za-z0-9.,;:!?()/%'\"+\- ]", " ", text)

    # Normalize spaces again
    text = re.sub(r"\s+", " ", text)

    return text.strip()


def safe_get(record, *keys):
    """
    Safely retrieve a value from a dictionary
    without causing KeyError.
    """
    for key in keys:
        if key in record:
            return record[key]

    # Case-insensitive search
    lower_map = {str(k).lower(): k for k in record.keys()}

    for key in keys:
        if key.lower() in lower_map:
            return record[lower_map[key.lower()]]

    return ""


def request_with_retry(
    url,
    method="GET",
    params=None,
    data=None,
    headers=None,
    timeout=60,
    retries=3
):
    """
    Request helper with retry mechanism.
    """

    last_error = None

    for attempt in range(1, retries + 1):

        try:

            if method.upper() == "POST":
                response = requests.post(
                    url,
                    params=params,
                    data=data,
                    headers=headers,
                    timeout=timeout
                )
            else:
                response = requests.get(
                    url,
                    params=params,
                    headers=headers,
                    timeout=timeout
                )

            response.raise_for_status()

            return response

        except Exception as e:

            last_error = e

            print(
                f"   Attempt {attempt}/{retries} failed: {e}"
            )

            if attempt < retries:
                time.sleep(2)

    raise RuntimeError(
        f"Request failed after {retries} attempts: {last_error}"
    )


# ============================================================
# 3. WHO DISEASE OUTBREAK NEWS
# ============================================================

print("\n" + "=" * 70)
print("[2/8] COLLECTING WHO DISEASE OUTBREAK NEWS")
print("=" * 70)

who_urls = [
    "https://www.who.int/api/news/diseaseoutbreaknews",
    "https://www.who.int/api/emergencies/diseaseoutbreaknews"
]

who_data = None
who_error = None

for who_url in who_urls:

    try:

        print("\nTrying:", who_url)

        response = request_with_retry(
            who_url,
            timeout=60,
            retries=3
        )

        raw_json = response.json()

        # WHO API may return a list directly or inside a container
        if isinstance(raw_json, list):
            who_data = raw_json

        elif isinstance(raw_json, dict):

            if isinstance(raw_json.get("value"), list):
                who_data = raw_json["value"]

            elif isinstance(raw_json.get("items"), list):
                who_data = raw_json["items"]

            elif isinstance(raw_json.get("results"), list):
                who_data = raw_json["results"]

            else:
                # Search dictionary values for a list
                found_list = None

                for value in raw_json.values():

                    if isinstance(value, list):
                        found_list = value
                        break

                who_data = found_list

        if who_data is not None and len(who_data) > 0:

            print("WHO records received:", len(who_data))
            break

    except Exception as e:

        who_error = e
        print("WHO endpoint failed:", e)


if who_data is None or len(who_data) == 0:

    raise RuntimeError(
        "WHO API returned no records. "
        "Please rerun this cell after a few minutes."
    )


# Limit to 50 records for the project prototype
who_data = who_data[:50]

who_rows = []

for i, item in enumerate(who_data):

    if not isinstance(item, dict):
        continue

    publication_date = safe_get(
        item,
        "PublicationDate",
        "publicationDate",
        "PublicationDateAndTime"
    )

    title = safe_get(
        item,
        "Title",
        "title"
    )

    overview = safe_get(
        item,
        "Overview",
        "overview"
    )

    epidemiology = safe_get(
        item,
        "Epidemiology",
        "epidemiology"
    )

    assessment = safe_get(
        item,
        "Assessment",
        "assessment"
    )

    advice = safe_get(
        item,
        "Advice",
        "advice"
    )

    response_text = safe_get(
        item,
        "Response",
        "response"
    )

    further_information = safe_get(
        item,
        "FurtherInformation",
        "furtherInformation"
    )

    don_id = safe_get(
        item,
        "DonId",
        "DONId",
        "donId"
    )

    provider = safe_get(
        item,
        "Provider",
        "provider"
    )

    who_rows.append({
        "id": f"WHO_{i+1}",
        "publication_date": publication_date,
        "title": title,
        "overview": overview,
        "epidemiology": epidemiology,
        "assessment": assessment,
        "advice": advice,
        "response": response_text,
        "further_information": further_information,
        "don_id": don_id,
        "provider": provider
    })


who_df = pd.DataFrame(who_rows)

who_path = os.path.join(
    RAW_DIR,
    "who_outbreaks.csv"
)

who_df.to_csv(
    who_path,
    index=False,
    encoding="utf-8"
)

print("\nWHO records saved:", len(who_df))
print("File:", who_path)

print("\nWHO sample:")
display(
    who_df[
        [
            "publication_date",
            "title",
            "provider"
        ]
    ].head(5)
)


# ============================================================
# 4. PUBMED
# ============================================================

print("\n" + "=" * 70)
print("[3/8] COLLECTING PUBMED ARTICLES")
print("=" * 70)

PUBMED_SEARCH_URL = (
    "https://eutils.ncbi.nlm.nih.gov/"
    "entrez/eutils/esearch.fcgi"
)

PUBMED_FETCH_URL = (
    "https://eutils.ncbi.nlm.nih.gov/"
    "entrez/eutils/efetch.fcgi"
)

PUBMED_QUERY = (
    '("One Health"[Title/Abstract] OR '
    '"zoonotic disease"[Title/Abstract] OR '
    '"zoonoses"[Title/Abstract] OR '
    '"emerging infectious disease"[Title/Abstract] OR '
    '"spillover"[Title/Abstract])'
)

# Search PubMed
search_params = {
    "db": "pubmed",
    "term": PUBMED_QUERY,
    "retmode": "xml",
    "retmax": 500,
    "tool": "OneHealthNexus",
    "email": "onehealthnexus@example.com"
}

search_response = request_with_retry(
    PUBMED_SEARCH_URL,
    params=search_params,
    timeout=60,
    retries=3
)

search_root = ET.fromstring(
    search_response.content
)

pmid_list = [
    element.text
    for element in search_root.findall(".//Id")
    if element.text
]

count_element = search_root.find(".//Count")

pubmed_total = (
    int(count_element.text)
    if count_element is not None
    else len(pmid_list)
)

print("Total PubMed search results:", pubmed_total)
print("PMIDs selected:", len(pmid_list))


# ------------------------------------------------------------
# Fetch PubMed records in small POST batches
# ------------------------------------------------------------

def fetch_pubmed_batch(pmids):

    if not pmids:
        return []

    fetch_data = {
        "db": "pubmed",
        "id": ",".join(pmids),
        "retmode": "xml",
        "rettype": "abstract",
        "tool": "OneHealthNexus",
        "email": "onehealthnexus@example.com"
    }

    response = request_with_retry(
        PUBMED_FETCH_URL,
        method="POST",
        data=fetch_data,
        timeout=120,
        retries=3
    )

    root = ET.fromstring(
        response.content
    )

    records = []

    for article in root.findall(".//PubmedArticle"):

        pmid_element = article.find(".//PMID")

        if pmid_element is None:
            continue

        pmid = pmid_element.text or ""

        # -------------------------------
        # Title
        # -------------------------------

        title_element = article.find(
            ".//ArticleTitle"
        )

        title = ""

        if title_element is not None:
            title = "".join(
                title_element.itertext()
            )

        # -------------------------------
        # Abstract
        # -------------------------------

        abstract_parts = []

        for abstract_element in article.findall(
            ".//Abstract/AbstractText"
        ):

            label = abstract_element.attrib.get(
                "Label",
                ""
            )

            text = "".join(
                abstract_element.itertext()
            )

            if label:
                abstract_parts.append(
                    f"{label}: {text}"
                )
            else:
                abstract_parts.append(
                    text
                )

        abstract = " ".join(
            abstract_parts
        )

        # -------------------------------
        # Publication year
        # -------------------------------

        year = ""

        year_candidates = [
            article.find(".//PubDate/Year"),
            article.find(".//PubDate/MedlineDate")
        ]

        for element in year_candidates:

            if element is not None and element.text:

                match = re.search(
                    r"(19|20)\d{2}",
                    element.text
                )

                if match:
                    year = match.group(0)
                    break

        # -------------------------------
        # Journal
        # -------------------------------

        journal_element = article.find(
            ".//Journal/Title"
        )

        journal = (
            journal_element.text
            if journal_element is not None
            else ""
        )

        records.append({
            "pmid": pmid,
            "title": title,
            "abstract": abstract,
            "year": year,
            "journal": journal,
            "source": "PubMed"
        })

    return records


pubmed_records = []

batch_size = 100

for start in range(
    0,
    len(pmid_list),
    batch_size
):

    batch = pmid_list[
        start:start + batch_size
    ]

    print(
        f"Fetching PubMed records "
        f"{start + 1} - "
        f"{min(start + batch_size, len(pmid_list))}"
    )

    try:

        batch_records = fetch_pubmed_batch(
            batch
        )

        pubmed_records.extend(
            batch_records
        )

    except Exception as e:

        print(
            "Batch failed:",
            e
        )

    time.sleep(0.5)


pubmed_df = pd.DataFrame(
    pubmed_records
)

# Remove duplicate PMIDs
if not pubmed_df.empty:

    pubmed_df = pubmed_df.drop_duplicates(
        subset=["pmid"]
    )

pubmed_path = os.path.join(
    RAW_DIR,
    "pubmed.csv"
)

pubmed_df.to_csv(
    pubmed_path,
    index=False,
    encoding="utf-8"
)

print("\nPubMed records saved:", len(pubmed_df))
print("File:", pubmed_path)

print("\nPubMed sample:")
display(
    pubmed_df.head(5)
)


# ============================================================
# 5. GBIF
# ============================================================

print("\n" + "=" * 70)
print("[4/8] COLLECTING GBIF ANIMAL OCCURRENCES")
print("=" * 70)

GBIF_URL = (
    "https://api.gbif.org/v1/occurrence/search"
)

gbif_params = {
    "limit": 300,
    "kingdom": "Animalia",
    "country": "IN",
    "has_coordinate": "true"
}

gbif_response = request_with_retry(
    GBIF_URL,
    params=gbif_params,
    timeout=120,
    retries=3
)

gbif_json = gbif_response.json()

gbif_results = gbif_json.get(
    "results",
    []
)

gbif_df = pd.DataFrame(
    gbif_results
)

print(
    "Original GBIF records:",
    len(gbif_df)
)


# ------------------------------------------------------------
# Extract required GBIF fields safely
# ------------------------------------------------------------

def gbif_column(df, column):
    if column in df.columns:
        return df[column]
    return pd.Series(
        [""] * len(df)
    )


gbif_clean = pd.DataFrame({

    "gbif_id":
        gbif_column(gbif_df, "key"),

    "species":
        gbif_column(gbif_df, "species"),

    "scientific_name":
        gbif_column(
            gbif_df,
            "scientificName"
        ),

    "kingdom":
        gbif_column(gbif_df, "kingdom"),

    "phylum":
        gbif_column(gbif_df, "phylum"),

    "class":
        gbif_column(gbif_df, "class"),

    "order":
        gbif_column(gbif_df, "order"),

    "family":
        gbif_column(gbif_df, "family"),

    "genus":
        gbif_column(gbif_df, "genus"),

    "country":
        gbif_column(gbif_df, "country"),

    "state":
        gbif_column(
            gbif_df,
            "stateProvince"
        ),

    "locality":
        gbif_column(
            gbif_df,
            "locality"
        ),

    "latitude":
        gbif_column(
            gbif_df,
            "decimalLatitude"
        ),

    "longitude":
        gbif_column(
            gbif_df,
            "decimalLongitude"
        ),

    "year":
        gbif_column(
            gbif_df,
            "year"
        ),

    "event_date":
        gbif_column(
            gbif_df,
            "eventDate"
        ),

    "dataset":
        gbif_column(
            gbif_df,
            "datasetName"
        )
})

# Remove duplicate records
gbif_clean = gbif_clean.drop_duplicates(
    subset=["gbif_id"]
)

gbif_path = os.path.join(
    RAW_DIR,
    "gbif.csv"
)

gbif_clean.to_csv(
    gbif_path,
    index=False,
    encoding="utf-8"
)

print(
    "Clean GBIF records:",
    len(gbif_clean)
)

print(
    "File:",
    gbif_path
)

print("\nGBIF sample:")
display(
    gbif_clean.head(5)
)


# ============================================================
# 6. CREATE COMMON CLEANED DOCUMENT DATASET
# ============================================================

print("\n" + "=" * 70)
print("[5/8] CREATING CLEANED DOCUMENT DATASET")
print("=" * 70)


# ------------------------------------------------------------
# WHO → common document format
# ------------------------------------------------------------

who_documents = []

for _, row in who_df.iterrows():

    text_parts = [
        row.get("title", ""),
        row.get("overview", ""),
        row.get("epidemiology", ""),
        row.get("assessment", ""),
        row.get("advice", ""),
        row.get("response", ""),
        row.get("further_information", "")
    ]

    full_text = " ".join(
        [
            str(x)
            for x in text_parts
            if pd.notna(x)
        ]
    )

    who_documents.append({

        "document_id":
            row["id"],

        "text":
            full_text,

        "date":
            row["publication_date"],

        "source":
            "WHO",

    })


who_documents_df = pd.DataFrame(
    who_documents
)


# ------------------------------------------------------------
# PubMed → common document format
# ------------------------------------------------------------

pubmed_documents = []

for _, row in pubmed_df.iterrows():

    title = row.get(
        "title",
        ""
    )

    abstract = row.get(
        "abstract",
        ""
    )

    full_text = (
        str(title)
        + " "
        + str(abstract)
    )

    pubmed_documents.append({

        "document_id":
            "PUBMED_" + str(
                row["pmid"]
            ),

        "text":
            full_text,

        "date":
            row.get(
                "year",
                ""
            ),

        "source":
            "PubMed",

    })


pubmed_documents_df = pd.DataFrame(
    pubmed_documents
)


# ------------------------------------------------------------
# Combine WHO + PubMed
# ------------------------------------------------------------

documents = pd.concat(
    [
        who_documents_df,
        pubmed_documents_df
    ],
    ignore_index=True
)


# ------------------------------------------------------------
# Clean text
# ------------------------------------------------------------

documents["clean_text"] = (
    documents["text"]
    .fillna("")
    .apply(clean_text)
)


# Remove empty documents

documents = documents[
    documents["clean_text"].str.len() > 0
].copy()


# Text length

documents["text_length"] = (
    documents["clean_text"]
    .str.len()
)


# Remove duplicate text

documents = documents.drop_duplicates(
    subset=["clean_text"]
)


# Reset index

documents = documents.reset_index(
    drop=True
)


cleaned_documents_path = os.path.join(
    PROCESSED_DIR,
    "cleaned_documents.csv"
)

documents.to_csv(
    cleaned_documents_path,
    index=False,
    encoding="utf-8"
)

print(
    "Cleaned documents:",
    len(documents)
)

print(
    "File:",
    cleaned_documents_path
)

print("\nCleaned document sample:")
display(
    documents.head(5)
)


# ============================================================
# 7. CLEAN GBIF
# ============================================================

print("\n" + "=" * 70)
print("[6/8] SAVING CLEANED GBIF DATA")
print("=" * 70)

gbif_clean_path = os.path.join(
    PROCESSED_DIR,
    "cleaned_gbif.csv"
)

gbif_clean.to_csv(
    gbif_clean_path,
    index=False,
    encoding="utf-8"
)

print(
    "Original GBIF:",
    len(gbif_df)
)

print(
    "Clean GBIF:",
    len(gbif_clean)
)

print(
    "File:",
    gbif_clean_path
)


# ============================================================
# 8. CREATE DATA SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("[7/8] CREATING DATA SUMMARY")
print("=" * 70)

summary = pd.DataFrame({

    "dataset": [
        "WHO Disease Outbreak News",
        "PubMed",
        "GBIF",
        "Combined Documents"
    ],

    "records": [
        len(who_df),
        len(pubmed_df),
        len(gbif_clean),
        len(documents)
    ],

    "file": [
        "data/raw/who_outbreaks.csv",
        "data/raw/pubmed.csv",
        "data/raw/gbif.csv",
        "data/processed/cleaned_documents.csv"
    ]
})

summary_path = os.path.join(
    PROCESSED_DIR,
    "data_summary.csv"
)

summary.to_csv(
    summary_path,
    index=False,
    encoding="utf-8"
)

display(summary)


# ============================================================
# 9. FINAL VALIDATION
# ============================================================

print("\n" + "=" * 70)
print("[8/8] FINAL VALIDATION")
print("=" * 70)

required_files = [

    os.path.join(
        RAW_DIR,
        "who_outbreaks.csv"
    ),

    os.path.join(
        RAW_DIR,
        "pubmed.csv"
    ),

    os.path.join(
        RAW_DIR,
        "gbif.csv"
    ),

    os.path.join(
        PROCESSED_DIR,
        "cleaned_documents.csv"
    ),

    os.path.join(
        PROCESSED_DIR,
        "cleaned_gbif.csv"
    ),

    os.path.join(
        PROCESSED_DIR,
        "data_summary.csv"
    )
]

all_files_ok = True

for path in required_files:

    exists = os.path.exists(path)

    size = (
        os.path.getsize(path)
        if exists
        else 0
    )

    print(
        f"{'✅' if exists else '❌'} "
        f"{path} "
        f"({size:,} bytes)"
    )

    if not exists:
        all_files_ok = False


# ============================================================
# 10. SHOW FINAL DATA INFORMATION
# ============================================================

print("\n" + "=" * 70)
print("FINAL DATASET INFORMATION")
print("=" * 70)

print(
    "\nWHO:",
    len(who_df),
    "records"
)

print(
    "PubMed:",
    len(pubmed_df),
    "records"
)

print(
    "GBIF:",
    len(gbif_clean),
    "records"
)

print(
    "Combined documents:",
    len(documents),
    "records"
)


print("\nDocument sources:")
print(
    documents["source"]
    .value_counts()
)


print("\nDocument columns:")
print(
    list(documents.columns)
)


print("\nGBIF columns:")
print(
    list(gbif_clean.columns)
)


# ============================================================
# 11. CREATE ZIP FILE
# ============================================================

ZIP_PATH = "/content/OneHealth_Nexus_data.zip"

if os.path.exists(ZIP_PATH):
    os.remove(ZIP_PATH)

with zipfile.ZipFile(
    ZIP_PATH,
    "w",
    zipfile.ZIP_DEFLATED
) as zipf:

    for root, dirs, files_list in os.walk(
        BASE_DIR
    ):

        for file_name in files_list:

            full_path = os.path.join(
                root,
                file_name
            )

            relative_path = os.path.relpath(
                full_path,
                "/content"
            )

            zipf.write(
                full_path,
                relative_path
            )


print("\n" + "=" * 70)

if all_files_ok:

    print(
        "✅ COMPLETE PIPELINE FINISHED SUCCESSFULLY"
    )

    print(
        "\nZIP created:",
        ZIP_PATH
    )

    print(
        "\nYour project data is ready."
    )

else:

    print(
        "❌ SOME FILES ARE MISSING."
    )

    raise RuntimeError(
        "Pipeline validation failed."
    )


# ============================================================
# 12. DOWNLOAD ZIP
# ============================================================

print("\nDownloading project data ZIP...")

files.download(
    ZIP_PATH
)

        ONEHEALTH NEXUS - DATA PIPELINE

[1/8] Project folders created.
BASE: /content/OneHealth_Nexus

[2/8] COLLECTING WHO DISEASE OUTBREAK NEWS

Trying: https://www.who.int/api/news/diseaseoutbreaknews
WHO records received: 50

WHO records saved: 50
File: /content/OneHealth_Nexus/data/raw/who_outbreaks.csv

WHO sample:


,publication_date,title,provider
0,2006-03-20T00:00:00Z,Avian influenza – situation in Egypt,dynamicProvider372
1,2009-02-09T00:00:00Z,Avian influenza - situation in Egypt - update 4,dynamicProvider372
2,2022-07-28T09:16:58Z,HIV cases–Pakistan,dynamicProvider372
3,2013-06-23T00:00:00Z,Middle East respiratory syndrome coronavirus (...,dynamicProvider372
4,2024-03-07T13:06:11Z,Influenza A (H5N1) virus - China,dynamicProvider372



[3/8] COLLECTING PUBMED ARTICLES
Total PubMed search results: 45521
PMIDs selected: 500
Fetching PubMed records 1 - 100
Fetching PubMed records 101 - 200
Fetching PubMed records 201 - 300
Fetching PubMed records 301 - 400
Fetching PubMed records 401 - 500

PubMed records saved: 500
File: /content/OneHealth_Nexus/data/raw/pubmed.csv

PubMed sample:


,pmid,title,abstract,year,journal,source
0,42790309,Economic burden and cost-effectiveness of bruc...,Bovine brucellosis caused by Brucella abortus ...,2026,Preventive veterinary medicine,PubMed
1,42790045,Microbial community structure and antibiotic r...,Karst aquifers are vulnerable drinking water r...,2026,Journal of hazardous materials,PubMed
2,42789295,Climate-driven host ecology as a framework for...,Mechanistic frameworks linking climate to zoon...,2026,Proceedings of the National Academy of Science...,PubMed
3,42789120,Forecast-Driven Dynamic Physician Staffing in ...,Emergency department crowding is a persistent ...,2026,Journal of medical systems,PubMed
4,42788961,Interfacial Oxygen Spillover Between Ru Single...,The sluggish oxygen evolution reaction (OER) r...,2026,"Small (Weinheim an der Bergstrasse, Germany)",PubMed



[4/8] COLLECTING GBIF ANIMAL OCCURRENCES
Original GBIF records: 300
Clean GBIF records: 300
File: /content/OneHealth_Nexus/data/raw/gbif.csv

GBIF sample:


...


[6/8] SAVING CLEANED GBIF DATA
Original GBIF: 300
Clean GBIF: 300
File: /content/OneHealth_Nexus/data/processed/cleaned_gbif.csv

[7/8] CREATING DATA SUMMARY


,dataset,records,file
0,WHO Disease Outbreak News,50,data/raw/who_outbreaks.csv
1,PubMed,500,data/raw/pubmed.csv
2,GBIF,300,data/raw/gbif.csv
3,Combined Documents,550,data/processed/cleaned_documents.csv



[8/8] FINAL VALIDATION
✅ /content/OneHealth_Nexus/data/raw/who_outbreaks.csv (235,508 bytes)
✅ /content/OneHealth_Nexus/data/raw/pubmed.csv (993,875 bytes)
✅ /content/OneHealth_Nexus/data/raw/gbif.csv (67,538 bytes)
✅ /content/OneHealth_Nexus/data/processed/cleaned_documents.csv (2,339,478 bytes)
✅ /content/OneHealth_Nexus/data/processed/cleaned_gbif.csv (67,538 bytes)
✅ /content/OneHealth_Nexus/data/processed/data_summary.csv (195 bytes)

FINAL DATASET INFORMATION

WHO: 50 records
PubMed: 500 records
GBIF: 300 records
Combined documents: 550 records

Document sources:
source
PubMed    500
WHO        50
Name: count, dtype: int64

Document columns:
['document_id', 'text', 'date', 'source', 'clean_text', 'text_length']

GBIF columns:
['gbif_id', 'species', 'scientific_name', 'kingdom', 'phylum', 'class', 'order', 'family', 'genus', 'country', 'state', 'locality', 'latitude', 'longitude', 'year', 'event_date', 'dataset']

✅ COMPLETE PIPELINE FINISHED SUCCESSFULLY

ZIP created: /content/O

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>